# Classical Rigid Docking Baseline

This standalone notebook demonstrates a standard "classical" docking approach. It takes a **single, rigid AlphaFold model** and performs docking exactly once.

This mimics what most researchers do when they use AutoDock Vina naively without ensembles (which SPADE attempts to fix). By running this, you can compare the single arbitrary score you get here against the robust `Ensemble Coverage` cluster scores produced by SPADE.

### 1. Install Dependencies

In [ ]:
!apt-get update && apt-get install -y swig
!pip install gemmi
!pip install --upgrade git+https://github.com/ChandraguptSharma07/Spade.git

### 2. Run Classical Docking

We use the SPADE internals to do the exact same AutoDock Vina procedure (target extraction, ligand prep, bounding box calculation), but we intentionally bypass the `EnsembleGenerator` step.

In [ ]:
import time
import numpy as np
from spade.core.structure import fetch_structure
from spade.core.ligand import prepare_ligand
from spade.core.docking import compute_bounding_box, _dock_single

# 1. Fetch rigid AlphaFold target (EGFR)
print("1. Fetching rigid AlphaFold target...")
structure = fetch_structure('P00533')
rigid_conformer = structure.atoms  # We use the raw structure directly

# 2. Prepare ONE ligand isomer
print("2. Preparing ONE ligand isomer...")
ERLOTINIB_SMILES = 'CCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOCCO'
# enumerate_stereo=False forces embedding the exact typed SMILES
ligands = prepare_ligand(ERLOTINIB_SMILES, ph=7.4, enumerate_stereo=False)
single_ligand = ligands[0]

# 3. Calculate bounding box on the rigid target
pocket_residues = np.arange(695, 720) # EGFR binding pocket
bbox = compute_bounding_box(rigid_conformer, pocket_residues)

# 4. Standard AutoDock Vina Run (1 Receptor vs 1 Ligand)
print("3. Running standard AutoDock Vina...")
t0 = time.perf_counter()
poses = _dock_single(
    conformer=rigid_conformer,
    ligand=single_ligand,
    bbox=bbox,
    exhaustiveness=8,
    n_poses=9,
    conf_idx=0
)
elapsed = time.perf_counter() - t0

# 5. Results
print("\n--- CLASSICAL DOCKING RESULTS ---")
print(f"Docking took {elapsed:.1f} seconds.")
print(f"Top Pose Score: {poses[0].score_kcal_mol} kcal/mol")